# RFP-Cap: Residual Feature Projection with Spatial Attention for Image Captioning

**Research notebook | IEEE SSCI 2027 paper companion — archived single-run experiment**  
**Contributors (SSCI 2027 paper author order):**  
1. **Christian Arthur** — University of Technology Sydney, Australia  
2. **Raymond Chiong** — University of New England, Australia  
3. **Isanka Dissanayake** — University of Technology Sydney, Australia  
4. **Raymond Halim** — University of Technology Sydney, Australia  
5. **Farshid Hajati** — University of New England, Australia  
**Scope:** spatial ResNet-50 + Bahdanau attention + LSTM versus the RFP-Cap dual-path feature-projection variant.



### A. Imports, vocabulary, artifacts, and dataset

In [ ]:
import os
import re
import json
import math
import time
import random
import pickle
from pathlib import Path
from collections import Counter, defaultdict

import nltk
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torchvision import transforms, models
from PIL import Image
from tqdm.auto import tqdm

os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

# ── Vocabulary — must match Phase 1 exactly so the pickle loads correctly ────
class Vocabulary:
    """Token ↔ index mapping. Special tokens always at fixed indices 0-3."""
    SPECIALS = ['<pad>', '<unk>', '<start>', '<end>']

    def __init__(self, min_freq: int = 5):
        self._min_freq = min_freq
        self._itos: list = []
        self._stoi: dict = {}

    def build(self, df):
        """Count tokens in df and keep those above min_freq."""
        counter = Counter(t for tokens in df['tokenized_caption'] for t in tokens)
        self._itos = list(self.SPECIALS)
        for tok, cnt in counter.most_common():
            if cnt < self._min_freq:
                break
            if tok not in self.SPECIALS:
                self._itos.append(tok)
        self._stoi = {t: i for i, t in enumerate(self._itos)}

    def __len__(self): return len(self._itos)

    @property
    def stoi(self):      return self._stoi
    @property
    def itos(self):      return self._itos
    @property
    def pad_idx(self):   return self._stoi['<pad>']
    @property
    def unk_idx(self):   return self._stoi['<unk>']
    @property
    def start_idx(self): return self._stoi['<start>']
    @property
    def end_idx(self):   return self._stoi['<end>']

    def encode(self, tokens: list) -> list:
        """Convert token strings to integer indices. Unknown tokens → UNK."""
        return [self._stoi.get(t, self.unk_idx) for t in tokens]

    def decode(self, ids: list, clean: bool = True) -> list:
        """Convert integer indices to token strings.
        If clean=True, strips PAD/START and stops at END."""
        tokens = [self._itos[i] for i in ids]
        if not clean:
            return tokens
        out = []
        for t in tokens:
            if t == '<end>':              break
            if t in ('<pad>', '<start>'): continue
            out.append(t)
        return out

In [ ]:
# Load Phase 1 artifacts 
ARTIFACT_PATH = Path('artifacts/phase1_artifacts.pkl')
if not ARTIFACT_PATH.exists():
    raise FileNotFoundError(
        f'Phase 1 artifact missing: {ARTIFACT_PATH}. Restore it alongside the VizWiz image data to rerun training.'
    )
with ARTIFACT_PATH.open('rb') as f:
    art = pickle.load(f)

vocab      = art['vocab']
VOCAB_SIZE = len(vocab)

# Device 
device = torch.device('cuda' if torch.cuda.is_available()
                      else 'mps' if torch.backends.mps.is_available()
                      else 'cpu')

# Reproducibility 
SEED = art['config'].get('seed', 29)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
if torch.backends.mps.is_available():
    torch.mps.manual_seed(SEED)  # best-effort reproducibility on Apple Silicon

print(f'Vocab size : {VOCAB_SIZE:,}')
print(f'Device     : {device}')
print(f'Seed       : {SEED}')
print(f'Splits     : train={len(art["train_ids"])} | val={len(art["val_ids"])} | test={len(art["test_ids"])}')
print('Note: MPS does not guarantee full determinism between runs.')

In [ ]:
# Load DataFrames and caption lookup
if 'df_train' in art:
    df_train = art['df_train']
    df_val   = art['df_val']
    df_test  = art['df_test']
    id_to_file = dict(zip(df_train['image_id'], df_train['file_name']))
    id_to_file.update(zip(df_val['image_id'], df_val['file_name']))
    id_to_file.update(zip(df_test['image_id'], df_test['file_name']))
    captions_by_image = defaultdict(list)
    for df in [df_train, df_val, df_test]:
        for _, row in df.iterrows():
            captions_by_image[row['image_id']].append(row['tokenized_caption'])
    captions_by_image = dict(captions_by_image)
    print('Loaded DataFrames from Phase 1 artifact.')
else:
    from torchtext.data.utils import get_tokenizer
    ANN_FILE  = Path('annotations/val.json')
    tokenizer = get_tokenizer('basic_english')
    MAX_LEN   = art['config']['max_len']
    def clean_caption(text):
        text = re.sub(r'[^\w\s]', '', text.lower().strip())
        return re.sub(r'\s+', ' ', text)
    def tokenize_caption(text):
        tokens = tokenizer(clean_caption(text))[:MAX_LEN - 2]
        return ['<start>'] + tokens + ['<end>']
    with open(ANN_FILE) as f:
        raw = json.load(f)
    id_to_file = {img['id']: img['file_name'] for img in raw['images']}
    captions_by_image = defaultdict(list)
    for ann in raw['annotations']:
        if ann.get('is_rejected') or ann.get('is_precanned'): continue
        captions_by_image[ann['image_id']].append(tokenize_caption(ann['caption']))
    captions_by_image = dict(captions_by_image)
    print('Rebuilt from val.json.')

MAX_LEN = art['config']['max_len']
print(f'Images with captions : {len(captions_by_image):,}')

In [ ]:
#  Image transforms 
IMAGENET_MEAN = art['config']['imagenet_mean']
IMAGENET_STD  = art['config']['imagenet_std']

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
print('Transforms ready.')

In [ ]:
#  Dataset and DataLoaders 
IMG_DIR = Path(art['config']['img_dir'])
assert IMG_DIR.exists(), f'Image directory not found: {IMG_DIR}'

class VizWizCaptionDataset(Dataset):
    """One sample = (image_tensor, caption_ids, length).
    Each image contributes one sample per caption (up to 5).
    Returns None for images that fail to load.
    """
    def __init__(self, image_ids, captions_by_image, id_to_file, vocab, img_dir, transform):
        self.samples    = [(iid, tok) for iid in image_ids
                           for tok in captions_by_image.get(iid, [])]
        self.id_to_file = id_to_file
        self.vocab      = vocab
        self.img_dir    = Path(img_dir)
        self.transform  = transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        iid, tokens = self.samples[idx]
        try:
            img = Image.open(self.img_dir / self.id_to_file[iid]).convert('RGB')
            img = self.transform(img)
        except (FileNotFoundError, OSError) as e:
            print(f'Warning: could not load {iid} — {e}')
            return None
        caption_ids = torch.tensor(self.vocab.encode(tokens), dtype=torch.long)
        return img, caption_ids, len(caption_ids)

def make_collate_fn(pad_idx):
    """Drops None samples, pads captions to batch max length."""
    def collate(batch):
        batch = [x for x in batch if x is not None]
        if not batch: return None, None, None
        imgs, caps, lengths = zip(*batch)
        return (torch.stack(imgs, 0),
                pad_sequence(caps, batch_first=True, padding_value=pad_idx),
                torch.tensor(lengths, dtype=torch.long))
    return collate

train_ids  = art['train_ids']
val_ids    = art['val_ids']
test_ids   = art['test_ids']
BATCH_SIZE = art['config']['batch_size']
NUM_WORKERS= art['config']['num_workers']
PIN_MEMORY = art['config']['pin_memory']
PAD_IDX    = vocab.pad_idx

collate_fn = make_collate_fn(PAD_IDX)
_kw = dict(num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, collate_fn=collate_fn)

train_ds = VizWizCaptionDataset(train_ids, captions_by_image, id_to_file, vocab, IMG_DIR, train_transform)
val_ds   = VizWizCaptionDataset(val_ids,   captions_by_image, id_to_file, vocab, IMG_DIR, eval_transform)
test_ds  = VizWizCaptionDataset(test_ids,  captions_by_image, id_to_file, vocab, IMG_DIR, eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  **_kw)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, **_kw)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, **_kw)

print(f'Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}')
_imgs, _caps, _lens = next(iter(train_loader))
print(f'Batch — images: {_imgs.shape}  captions: {_caps.shape}')
print(f'Decoded sample: {" ".join(vocab.decode(_caps[0].tolist()))}')

### B. Shared attention, LSTM decoder, and evaluation utilities

Both configurations use Bahdanau additive attention and an LSTM decoder *family*. The attention input and LSTM context dimensions differ (2,048 versus 512), so parameter counts are not matched.

In [ ]:
class BahdanauAttention(nn.Module):
    """Additive attention (Bahdanau et al., 2015).
    score(h, f_i) = v^T · tanh(W_enc·f_i + W_dec·h)
    Separate learned mappings allow encoder and hidden-state inputs of different dimensions.
    """
    def __init__(self, encoder_dim, hidden_dim, attn_dim):
        super().__init__()
        self.W_enc = nn.Linear(encoder_dim, attn_dim)
        self.W_dec = nn.Linear(hidden_dim,  attn_dim)
        self.v     = nn.Linear(attn_dim, 1, bias=False)

    def forward(self, features, hidden):
        energy  = torch.tanh(self.W_enc(features) + self.W_dec(hidden).unsqueeze(1))
        scores  = self.v(energy).squeeze(-1)
        alpha   = torch.softmax(scores, dim=1)
        context = (alpha.unsqueeze(-1) * features).sum(1)
        return context, alpha


class DecoderLSTM(nn.Module):
    """Single-layer LSTM decoder with Bahdanau attention.
    Teacher forcing during training: ground truth previous token fed at each step.
    Hidden/cell states initialised from mean spatial feature.
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim,
                 encoder_dim=2048, attn_dim=256, dropout=0.5):
        super().__init__()
        self.vocab_size  = vocab_size
        self.hidden_dim  = hidden_dim
        self.encoder_dim = encoder_dim
        self.embed     = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.attention = BahdanauAttention(encoder_dim, hidden_dim, attn_dim)
        self.lstm_cell = nn.LSTMCell(embed_dim + encoder_dim, hidden_dim)
        self.init_h    = nn.Linear(encoder_dim, hidden_dim)
        self.init_c    = nn.Linear(encoder_dim, hidden_dim)
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_dim, vocab_size)

    def init_hidden(self, features):
        """Initialise (h, c) from the mean of all spatial features."""
        mean = features.mean(dim=1)
        return torch.tanh(self.init_h(mean)), torch.tanh(self.init_c(mean))

    def forward(self, features, captions):
        """Teacher-forcing forward pass.
        features: [B, N, encoder_dim]  captions: [B, T]
        Returns logits [B, T-1, vocab_size].
        """
        B, T   = captions.shape
        h, c   = self.init_hidden(features)
        embeds = self.embed(captions[:, :-1])
        logits = torch.zeros(B, T - 1, self.vocab_size, device=features.device)
        for t in range(T - 1):
            context, _ = self.attention(features, h)
            x_t        = torch.cat([embeds[:, t], context], dim=1)
            h, c       = self.lstm_cell(x_t, (h, c))
            logits[:, t] = self.fc(self.dropout(h))
        return logits


print('Shared components ready: BahdanauAttention, DecoderLSTM')

In [ ]:
# Shared constants and training utilities 
CHECKPOINT_DIR = Path('checkpoints')
CHECKPOINT_DIR.mkdir(exist_ok=True)

# Hyperparameters 
EMBED_DIM   = 256
HIDDEN_DIM  = 512
ATTN_DIM    = 256
ENCODER_DIM = 2048  # Model 1 raw ResNet-50 feature dimension
DROPOUT     = 0.5
LR_DECODER  = 4e-4
LR_ENCODER  = 1e-4


def compute_loss(logits, captions, pad_idx, label_smoothing=0.1):
    """CrossEntropyLoss with label_smoothing=0.1.
    Prevents overconfidence on training captions, improving generalisation.
    Ignores <pad> positions so padding does not contribute to the gradient.
    """
    loss_fn = nn.CrossEntropyLoss(ignore_index=pad_idx, label_smoothing=label_smoothing)
    targets = captions[:, 1:]
    return loss_fn(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))


def train_one_epoch(model, loader, optimizer, pad_idx, device, epoch_num):
    """One full pass over the training set. Returns average loss."""
    model.train()
    total_loss, n_batches = 0.0, 0
    start = time.time()
    for batch_idx, (images, captions, _) in enumerate(loader):
        if images is None: continue
        images, captions = images.to(device), captions.to(device)
        logits = model(images, captions)
        loss   = compute_loss(logits, captions, pad_idx)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item()
        n_batches  += 1
        if (batch_idx + 1) % 100 == 0:
            print(f'  Epoch {epoch_num} | batch {batch_idx+1}/{len(loader)} | '
                  f'avg loss {total_loss/n_batches:.4f} | elapsed {time.time()-start:.1f}s')
    return total_loss / max(n_batches, 1)


@torch.no_grad()
def validate(model, loader, pad_idx, device):
    """Average validation loss. No gradients computed."""
    model.eval()
    total_loss, n_batches = 0.0, 0
    for images, captions, _ in loader:
        if images is None: continue
        images, captions = images.to(device), captions.to(device)
        total_loss += compute_loss(model(images, captions), captions, pad_idx).item()
        n_batches  += 1
    return total_loss / max(n_batches, 1)


def fit(model, train_loader, val_loader, optimizer, pad_idx, device,
        num_epochs, model_name, scheduler=None):
    """Full training loop. Saves best checkpoint by lowest val loss.
    Calls scheduler.step(val_loss) each epoch if provided.
    Returns history dict.
    """
    history = {'train_loss': [], 'val_loss': [], 'epoch_time': [], 'lr': []}
    best_val_loss, best_epoch = math.inf, -1
    for epoch in range(1, num_epochs + 1):
        print(f'\n{"="*60}')
        print(f'Epoch {epoch}/{num_epochs}  [{model_name}]')
        print('='*60)
        t0         = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, pad_idx, device, epoch)
        val_loss   = validate(model, val_loader, pad_idx, device)
        epoch_time = time.time() - t0
        current_lr = optimizer.param_groups[0]['lr']
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['epoch_time'].append(epoch_time)
        history['lr'].append(current_lr)
        print(f'  Train loss : {train_loss:.4f}')
        print(f'  Val loss   : {val_loss:.4f}')
        print(f'  LR         : {current_lr:.2e}')
        print(f'  Epoch time : {epoch_time:.1f}s')
        if scheduler is not None:
            scheduler.step(val_loss)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch    = epoch
            ckpt_path     = CHECKPOINT_DIR / f'{model_name}_best.pt'
            torch.save({
                'epoch': epoch, 'val_loss': val_loss, 'train_loss': train_loss,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
            }, ckpt_path)
            print(f'  ✓ New best — saved to {ckpt_path}')
    print(f'\nDone. Best val loss {best_val_loss:.4f} at epoch {best_epoch}')
    return history


@torch.no_grad()
def generate_caption_beam(model, image, vocab, max_len=20, beam_size=5, device='cpu'):
    """Beam search decoding. Maintains beam_size candidate sequences at each step.
    Explores multiple next-token candidates; does not guarantee reduced repetition.
    """
    model.eval()
    img   = image.unsqueeze(0).to(device)
    feats = model.encoder(img)
    h, c  = model.decoder.init_hidden(feats)
    beams, completed = [(0.0, [vocab.start_idx], h, c)], []
    for _ in range(max_len):
        candidates = []
        for score, tokens, h, c in beams:
            if tokens[-1] == vocab.end_idx:
                completed.append((score, tokens)); continue
            word   = torch.tensor([tokens[-1]], device=device)
            embed  = model.decoder.embed(word)
            ctx, _ = model.decoder.attention(feats, h)
            x_t    = torch.cat([embed, ctx], dim=1)
            h_new, c_new = model.decoder.lstm_cell(x_t, (h, c))
            logits = model.decoder.fc(model.decoder.dropout(h_new))
            log_probs = torch.log_softmax(logits, dim=-1).squeeze(0)
            top_log_probs, top_ids = log_probs.topk(beam_size)
            for lp, tid in zip(top_log_probs.tolist(), top_ids.tolist()):
                candidates.append((score + lp, tokens + [tid], h_new, c_new))
        if not candidates: break
        beams = sorted(candidates, key=lambda x: x[0], reverse=True)[:beam_size]
    completed += [(s, t) for s, t, _, _ in beams]
    if not completed: return []
    return vocab.decode(max(completed, key=lambda x: x[0])[1])


def evaluate_model(model, test_ids, captions_by_image, id_to_file,
                   vocab, IMG_DIR, eval_transform, device, model_name):
    """Generate captions and compute NLTK BLEU and NLTK METEOR.
    The CIDEr-labeled historical statistic is a custom unigram proxy; NOT official CIDEr-D.
    Returns predictions, references, and diagnostic metrics for this custom split.
    """
    nltk.download('wordnet',   quiet=True)
    nltk.download('omw-1.4',  quiet=True)
    nltk.download('punkt_tab', quiet=True)
    from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
    from nltk.translate.meteor_score import meteor_score

    print(f'Generating captions for {model_name} (beam size=5)...')
    preds, refs = {}, {}
    for iid in tqdm(test_ids):
        if iid not in captions_by_image or not captions_by_image[iid]: continue
        try:
            img_tensor = eval_transform(Image.open(IMG_DIR / id_to_file[iid]).convert('RGB'))
        except (FileNotFoundError, OSError): continue
        preds[iid] = generate_caption_beam(model, img_tensor, vocab, beam_size=5, device=device)
        refs[iid]  = [[t for t in c if t not in ('<start>','<end>','<pad>')]
                      for c in captions_by_image[iid]]

    hyps = list(preds.values())
    rfs  = list(refs.values())
    smooth = SmoothingFunction().method1
    bleu1 = corpus_bleu(rfs, hyps, weights=(1,0,0,0),        smoothing_function=smooth)
    bleu2 = corpus_bleu(rfs, hyps, weights=(.5,.5,0,0),      smoothing_function=smooth)
    bleu3 = corpus_bleu(rfs, hyps, weights=(1/3,1/3,1/3,0),  smoothing_function=smooth)
    bleu4 = corpus_bleu(rfs, hyps, weights=(.25,.25,.25,.25), smoothing_function=smooth)
    meteor = np.mean([meteor_score(r, h) for h, r in zip(hyps, rfs) if h])
    
    # Nonstandard unigram TF-IDF proxy; DO NOT cite as CIDEr-D or benchmark CIDEr. 
    from math import log
    N  = len(hyps)
    df = Counter()
    for r in rfs: df.update(set(w for ref in r for w in ref))
    cider_scores = []
    for hyp, r in zip(hyps, rfs):
        if not hyp: cider_scores.append(0.0); continue
        hc = Counter(hyp); rc = Counter(w for ref in r for w in ref)
        cider_scores.append(sum(
            (cnt/len(hyp))*(rc.get(w,0)/max(sum(rc.values()),1))*log((N+1)/(df.get(w,0)+1))
            for w, cnt in hc.items()))
    cider = float(np.mean(cider_scores))

    metrics = {
        'BLEU-1': round(bleu1, 4),
        'BLEU-2': round(bleu2, 4),
        'BLEU-3': round(bleu3, 4),
        'BLEU-4': round(bleu4, 4),
        'METEOR': round(meteor, 4),
        'CIDEr proxy':  round(cider, 4),
    }

    print(f'\n{"="*50}')
    print(f' {model_name}')
    print('='*50)
    for k, v in metrics.items():
        print(f'  {k:<8}: {v:.4f}')
    print('='*50)
    print(f'Evaluated on {len(hyps):,} test images')
    return preds, refs, metrics


# Shared good/poor prediction helper 
STOP_WORDS = {'a','an','the','is','are','of','to','in','on',
              'with','and','it','this','that','at','by','for'}

def is_good_prediction(pred_tokens, ref_list, min_overlap=2):
    """Returns True if pred shares at least min_overlap content words with any reference.
    Stop words excluded so generic words do not count as meaningful matches.
    """
    pred_set = set(pred_tokens) - STOP_WORDS
    for ref in ref_list:
        if len(pred_set & (set(ref) - STOP_WORDS)) >= min_overlap:
            return True
    return False


print('Shared utilities ready.')    

### C Baseline — ResNet-50 + additive attention + LSTM

In [ ]:
class EncoderResNet50(nn.Module):
    """ResNet-50 encoder returning spatial features [B, 49, 2048].
    layer4 unfrozen for VizWiz domain adaptation.
    """
    def __init__(self, fine_tune_layer4=True):
        super().__init__()
        resnet      = models.resnet50(weights='IMAGENET1K_V2')
        modules     = list(resnet.children())[:-2]
        self.resnet = nn.Sequential(*modules)
        for p in self.resnet.parameters():
            p.requires_grad = False
        if fine_tune_layer4:
            for p in self.resnet[7].parameters():  # layer4 is index 7
                p.requires_grad = True

    def forward(self, images):
        features = self.resnet(images)         # [B, 2048, 7, 7]
        B, C, H, W = features.shape
        return features.view(B, C, H*W).permute(0, 2, 1)  # [B, 49, 2048]


class CaptionModelResNet50(nn.Module):
    """Model 1: ResNet-50 encoder + single LSTM decoder + Bahdanau attention."""
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=512,
                 encoder_dim=2048, attn_dim=256, dropout=0.5):
        super().__init__()
        self.encoder = EncoderResNet50(fine_tune_layer4=True)
        self.decoder = DecoderLSTM(vocab_size, embed_dim, hidden_dim,
                                   encoder_dim, attn_dim, dropout)
    def forward(self, images, captions):
        return self.decoder(self.encoder(images), captions)


torch.manual_seed(SEED)
model1 = CaptionModelResNet50(
    vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM,
    encoder_dim=ENCODER_DIM, attn_dim=ATTN_DIM, dropout=DROPOUT,
).to(device)

total_m1     = sum(p.numel() for p in model1.parameters())
trainable_m1 = sum(p.numel() for p in model1.parameters() if p.requires_grad)
print(f'Model 1 — ResNet-50 + Single LSTM + Bahdanau Attention')
print(f'Total params    : {total_m1:,}')
print(f'Trainable params: {trainable_m1:,}')
print(f'Frozen params   : {total_m1 - trainable_m1:,}')
model1.eval()
with torch.no_grad():
    _out = model1(_imgs.to(device), _caps.to(device))
print(f'Forward pass OK — logits shape: {_out.shape}')

#### Baseline training 

In [ ]:
M1_NUM_EPOCHS = 30

print(f'Optimizer  : Adam | Decoder LR: {LR_DECODER} | Encoder LR: {LR_ENCODER}')
print(f'Scheduler  : ReduceLROnPlateau (factor=0.5, patience=3)')
print(f'Label smooth: 0.1 | Grad clip: 5.0 | Epochs: {M1_NUM_EPOCHS}')

In [ ]:
#  Model 1 definition + optimizer + training 
torch.manual_seed(SEED)
model1 = CaptionModelResNet50(
    vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM,
    encoder_dim=ENCODER_DIM, attn_dim=ATTN_DIM, dropout=DROPOUT,
).to(device)

m1_encoder_params = [p for p in model1.encoder.parameters() if p.requires_grad]
m1_decoder_params = [p for p in model1.decoder.parameters() if p.requires_grad]
m1_optimizer = Adam([
    {'params': m1_decoder_params, 'lr': LR_DECODER},
    {'params': m1_encoder_params, 'lr': LR_ENCODER},
])
m1_scheduler = ReduceLROnPlateau(
    m1_optimizer, mode='min', factor=0.5, patience=3
)

total_m1     = sum(p.numel() for p in model1.parameters())
trainable_m1 = sum(p.numel() for p in model1.parameters() if p.requires_grad)
print(f'Model 1 — ResNet-50 + Single LSTM + Bahdanau Attention')
print(f'Total params    : {total_m1:,}')
print(f'Trainable params: {trainable_m1:,}')

print(f'\nTraining Model 1: {M1_NUM_EPOCHS} epochs | device: {device}')
history_m1 = fit(
    model1, train_loader, val_loader, m1_optimizer, PAD_IDX, device,
    num_epochs=M1_NUM_EPOCHS, model_name='model1_resnet50',
    scheduler=m1_scheduler
)

In [ ]:
# Model 1 training curves 
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
epochs = range(1, len(history_m1['train_loss']) + 1)
axes[0].plot(epochs, history_m1['train_loss'], 'b-o', label='Train', linewidth=2)
axes[0].plot(epochs, history_m1['val_loss'],   'r-s', label='Val',   linewidth=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Model 1 (ResNet-50) — Loss curves')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(epochs, history_m1['lr'], 'g-^', linewidth=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Learning rate')
axes[1].set_title('Model 1 — LR schedule')
axes[1].grid(True, alpha=0.3)
plt.suptitle('Model 1: ResNet-50 + LSTM + Bahdanau Attention', fontsize=12)
plt.tight_layout()
plt.savefig('report_figures/original_m1_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n{"Epoch":<8}{"Train Loss":<14}{"Val Loss":<14}{"LR":<12}{"Time (s)":<10}')
print('-'*56)
for i, (tl, vl, lr, t) in enumerate(
        zip(history_m1['train_loss'], history_m1['val_loss'],
            history_m1['lr'], history_m1['epoch_time']), 1):
    print(f'{i:<8}{tl:<14.4f}{vl:<14.4f}{lr:<12.2e}{t:<10.1f}')
best_e1 = history_m1['val_loss'].index(min(history_m1['val_loss'])) + 1
print(f'\nBest val loss: {min(history_m1["val_loss"]):.4f} at epoch {best_e1}')

#### Baseline checkpoint and evaluation

In [ ]:
ckpt_m1 = torch.load(
    CHECKPOINT_DIR / 'model1_resnet50_best.pt',
    map_location=device, weights_only=False
)
model1.load_state_dict(ckpt_m1['model_state_dict'])
model1.eval()
print(f'Loaded Model 1 checkpoint — epoch {ckpt_m1["epoch"]} | val loss {ckpt_m1["val_loss"]:.4f}')

In [ ]:
m1_preds, m1_refs, m1_metrics = evaluate_model(
    model1, test_ids, captions_by_image, id_to_file,
    vocab, IMG_DIR, eval_transform, device,
    'Model 1 — ResNet-50 + Single LSTM + Bahdanau Attention'
)

In [ ]:
# Model 1 good vs poor predictions
good_iids_m1 = [iid for iid in m1_preds if is_good_prediction(m1_preds[iid], m1_refs[iid])]
bad_iids_m1  = [iid for iid in m1_preds if not is_good_prediction(m1_preds[iid], m1_refs[iid])]
print(f'Good: {len(good_iids_m1)} ({100*len(good_iids_m1)/len(m1_preds):.1f}%)  '
      f'Poor: {len(bad_iids_m1)} ({100*len(bad_iids_m1)/len(m1_preds):.1f}%)')

random.seed(SEED)
sample_good_m1 = random.sample(good_iids_m1, min(5, len(good_iids_m1)))
sample_bad_m1  = random.sample(bad_iids_m1,  min(5, len(bad_iids_m1)))

fig, axes = plt.subplots(5, 2, figsize=(16, 24))
for row_idx in range(5):
    for col_idx, (iid, label) in enumerate([
        (sample_good_m1[row_idx], 'HIGHER OVERLAP'), (sample_bad_m1[row_idx], 'LOWER OVERLAP')]):
        ax  = axes[row_idx, col_idx]
        img = Image.open(IMG_DIR / id_to_file[iid]).convert('RGB')
        ax.imshow(img); ax.axis('off')
        color = '#2e7d32' if label == 'HIGHER OVERLAP' else '#c62828'
        ax.set_title(f'[{label}]\nPred: {" ".join(m1_preds[iid]) or "(empty)"}\n'
                     f'Ref:  {" ".join(m1_refs[iid][0])}',
                     fontsize=9, loc='left', color=color)
plt.suptitle('Model 1 (ResNet-50) — Higher vs Lower Lexical Overlap', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('report_figures/original_m1_predictions.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Model 1 failure analysis ─
rep_count, short_count, generic_count = 0, 0, 0
for pred in m1_preds.values():
    counts = Counter(pred)
    if len(pred) < 4: short_count += 1
    if any(c >= 3 for c in counts.values()): rep_count += 1
    if len(pred) >= 2 and pred[0]=='a' and pred[1] in ('person','man','woman','picture','photo'):
        generic_count += 1
total   = len(m1_preds)
lengths = [len(p) for p in m1_preds.values()]
print(f'Model 1 Failure Analysis ({total:,} test samples):')
print(f'  Repetition (word ≥3×)     : {rep_count} ({100*rep_count/total:.1f}%)')
print(f'  Short captions (< 4 words) : {short_count} ({100*short_count/total:.1f}%)')
print(f'  Generic openers            : {generic_count} ({100*generic_count/total:.1f}%)')
print(f'  Caption lengths — min:{min(lengths)} max:{max(lengths)} '
      f'mean:{sum(lengths)/len(lengths):.1f} median:{sorted(lengths)[len(lengths)//2]}')

### D. RFP-Cap — dual-path residual feature projection

The RFP-Cap projection has a normalized nonlinear branch (Linear → LayerNorm → ReLU → Dropout) and an **unnormalized learned linear shortcut**. The resulting sum has 512 channels at each of 49 positions. The shortcut is not an identity mapping, and any efficiency claim requires profiling.

In [ ]:
class EncoderResNet50Projected(nn.Module):
    """ResNet-50 encoder with learned feature projection and residual connection.

    ResNet-50 output : [B, 49, 2048]
    Projected output : [B, 49, 512]

    The projection reduces encoder dimensionality before attention,
    changing the feature width. LayerNorm is applied only to the
    nonlinear branch; the learned projected shortcut is not normalized.
    """
    def __init__(self, fine_tune_layer4=True, proj_dropout=0.1):
        super().__init__()
        resnet      = models.resnet50(weights='IMAGENET1K_V2')
        modules     = list(resnet.children())[:-2]
        self.resnet = nn.Sequential(*modules)

        for p in self.resnet.parameters():
            p.requires_grad = False
        if fine_tune_layer4:
            for p in self.resnet[7].parameters():  # layer4 is index 7
                p.requires_grad = True

        # Single-step projection: 2048 → 512 with LayerNorm
        # Simpler than 2048→1024→512 — fewer params, same effect
        self.projection = nn.Sequential(
            nn.Linear(2048, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(proj_dropout)
        )

        # Learned dimensionality-matching shortcut (not an identity mapping)
        self.skip = nn.Linear(2048, 512, bias=False)

    def forward(self, images):
        features = self.resnet(images)                          # [B, 2048, 7, 7]
        B, C, H, W = features.shape
        features = features.view(B, C, H*W).permute(0, 2, 1)  # [B, 49, 2048]

        # Projected path + residual skip
        projected = self.projection(features)  # [B, 49, 512]
        skip      = self.skip(features)        # [B, 49, 512]
        return projected + skip                # [B, 49, 512]


class CaptionModelResNet50Projected(nn.Module):
    """Model 2: ResNet-50 + projection + residual + LayerNorm
    + single LSTM decoder + Bahdanau attention.
    """
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=512,
                 encoder_dim=512, attn_dim=256, dropout=0.5):
        super().__init__()
        self.encoder = EncoderResNet50Projected(fine_tune_layer4=True, proj_dropout=0.1)
        self.decoder = DecoderLSTM(vocab_size, embed_dim, hidden_dim,
                                   encoder_dim, attn_dim, dropout)

    def forward(self, images, captions):
        return self.decoder(self.encoder(images), captions)


MODEL2_ENCODER_DIM = 512

torch.manual_seed(SEED)
model2 = CaptionModelResNet50Projected(
    vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM,
    encoder_dim=MODEL2_ENCODER_DIM, attn_dim=ATTN_DIM, dropout=DROPOUT,
).to(device)

total_m2     = sum(p.numel() for p in model2.parameters())
trainable_m2 = sum(p.numel() for p in model2.parameters() if p.requires_grad)
print(f'Model 2 — ResNet-50 + Projection + Residual + LayerNorm + LSTM + Bahdanau Attention')
print(f'Total params    : {total_m2:,}')
print(f'Trainable params: {trainable_m2:,}')
print(f'Frozen params   : {total_m2 - trainable_m2:,}')
print(f'Param diff vs Model 1: {total_m2 - total_m1:,}')
model2.eval()
with torch.no_grad():
    _out2 = model2(_imgs.to(device), _caps.to(device))
print(f'Forward pass OK — logits shape: {_out2.shape}')

#### RFP-Cap training — source configuration (15 epochs)

In [ ]:
M2_NUM_EPOCHS = 15   

m2_encoder_params = [p for p in model2.encoder.parameters() if p.requires_grad]
m2_decoder_params = [p for p in model2.decoder.parameters() if p.requires_grad]

m2_optimizer = Adam([
    {'params': m2_decoder_params, 'lr': LR_DECODER},
    {'params': m2_encoder_params, 'lr': LR_ENCODER},
])
m2_scheduler = ReduceLROnPlateau(
    m2_optimizer, mode='min', factor=0.5, patience=3
)

print(f'Optimizer  : Adam')
print(f'Decoder LR : {LR_DECODER}')
print(f'Encoder LR : {LR_ENCODER}')
print(f'Scheduler  : ReduceLROnPlateau (factor=0.5, patience=3)')
print(f'Label smooth: 0.1 | Grad clip: 5.0 | Epochs: {M2_NUM_EPOCHS}')

In [ ]:
print(f'Training Model 2: {M2_NUM_EPOCHS} epochs | device: {device}')
history_m2 = fit(
    model2, train_loader, val_loader, m2_optimizer, PAD_IDX, device,
    num_epochs=M2_NUM_EPOCHS, model_name='model2_resnet50_projected',
    scheduler=m2_scheduler
)

In [ ]:
# Model 2 training curves overlaid with Model 1 for comparison 
epochs_m1 = range(1, len(history_m1['train_loss']) + 1)
epochs_m2 = range(1, len(history_m2['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(epochs_m2, history_m2['train_loss'], 'b-o', label='Train M2', linewidth=2)
axes[0].plot(epochs_m2, history_m2['val_loss'],   'r-s', label='Val M2',   linewidth=2)
axes[0].plot(epochs_m1, history_m1['train_loss'], 'b--', alpha=0.4, label='Train M1')
axes[0].plot(epochs_m1, history_m1['val_loss'],   'r--', alpha=0.4, label='Val M1')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Model 2 vs Model 1 — Loss curves')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(epochs_m2, history_m2['lr'], 'g-^', linewidth=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Learning rate')
axes[1].set_title('Model 2 — LR schedule (patience=3)')
axes[1].grid(True, alpha=0.3)
plt.suptitle('Model 2: ResNet-50 + Projection Layers + LayerNorm + Single LSTM + Bahdanau Attention', fontsize=12)
plt.tight_layout()
plt.savefig('report_figures/original_m2_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n{"Epoch":<8}{"Train Loss":<14}{"Val Loss":<14}{"LR":<12}{"Time (s)":<10}')
print('-'*56)
for i, (tl, vl, lr, t) in enumerate(
        zip(history_m2['train_loss'], history_m2['val_loss'],
            history_m2['lr'], history_m2['epoch_time']), 1):
    print(f'{i:<8}{tl:<14.4f}{vl:<14.4f}{lr:<12.2e}{t:<10.1f}')
best_e2 = history_m2['val_loss'].index(min(history_m2['val_loss'])) + 1
print(f'\nBest val loss: {min(history_m2["val_loss"]):.4f} at epoch {best_e2}')

#### RFP-Cap checkpoint and evaluation

In [ ]:
ckpt_m2 = torch.load(
    CHECKPOINT_DIR / 'model2_resnet50_projected_best.pt',
    map_location=device,
    weights_only=False
)
model2.load_state_dict(ckpt_m2['model_state_dict'])
model2.eval()
print(f'Loaded Model 2 checkpoint — epoch {ckpt_m2["epoch"]} | val loss {ckpt_m2["val_loss"]:.4f}')

In [ ]:
m2_preds, m2_refs, m2_metrics = evaluate_model(
    model2, test_ids, captions_by_image, id_to_file,
    vocab, IMG_DIR, eval_transform, device,
    'Model 2 — ResNet-50 + Projection Layers + LayerNorm + Single LSTM + Bahdanau Attention'
)

In [ ]:
#  Model 2 good vs poor predictions 
good_iids_m2 = [iid for iid in m2_preds if is_good_prediction(m2_preds[iid], m2_refs[iid])]
bad_iids_m2  = [iid for iid in m2_preds if not is_good_prediction(m2_preds[iid], m2_refs[iid])]
print(f'Good: {len(good_iids_m2)} ({100*len(good_iids_m2)/len(m2_preds):.1f}%)  '
      f'Poor: {len(bad_iids_m2)} ({100*len(bad_iids_m2)/len(m2_preds):.1f}%)')

random.seed(SEED)
sample_good_m2 = random.sample(good_iids_m2, min(5, len(good_iids_m2)))
sample_bad_m2  = random.sample(bad_iids_m2,  min(5, len(bad_iids_m2)))

fig, axes = plt.subplots(5, 2, figsize=(16, 24))
for row_idx in range(5):
    for col_idx, (iid, label) in enumerate([
        (sample_good_m2[row_idx], 'HIGHER OVERLAP'), (sample_bad_m2[row_idx], 'LOWER OVERLAP')]):
        ax  = axes[row_idx, col_idx]
        img = Image.open(IMG_DIR / id_to_file[iid]).convert('RGB')
        ax.imshow(img); ax.axis('off')
        color = '#2e7d32' if label == 'HIGHER OVERLAP' else '#c62828'
        ax.set_title(f'[{label}]\nPred: {" ".join(m2_preds[iid]) or "(empty)"}\n'
                     f'Ref:  {" ".join(m2_refs[iid][0])}',
                     fontsize=9, loc='left', color=color)
plt.suptitle('Model 2 (Projected ResNet-50) — Higher vs Lower Lexical Overlap', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('report_figures/original_m2_predictions.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Model 1 vs Model 2 — Final Comparison
print(f'\n{"="*55}')
print(f'  ResNet-Attn-LSTM vs RFP-Cap — Archived Single Run')
print(f'{"="*55}')
print(f'  {"Metric":<10} {"Model 1":>12} {"Model 2":>12} {"Δ":>8}')
print(f'  {"-"*44}')
for m in m1_metrics:
    d    = m2_metrics[m] - m1_metrics[m]
    sign = '+' if d >= 0 else ''
    print(f'  {m:<10} {m1_metrics[m]:>12.4f} {m2_metrics[m]:>12.4f} {sign}{d:>7.4f}')
print(f'{"="*55}')

## 4 · Optional report figure: token-level attention maps (requires saved models and VizWiz image files)

The paper discusses spatial attention maps, but **the upload does not contain the checkpoint and original images** needed to regenerate those maps. The following cell produces **actual token-level attention visualizations** after Model 2 is loaded. It does not construct artificial heatmaps. For camera-ready use, retain the image ID, generated tokens, checkpoint identifier, and whether the figure depicts mean or per-token attention.

In [ ]:
# Run only after the Phase 1 data and trained RFP-Cap checkpoint have been restored and loaded.
# This uses greedy decoding for interpretable per-token maps, not beam-5 caption scoring.
import torch.nn.functional as F

@torch.no_grad()
def greedy_caption_with_attention(model, image_tensor, vocab, device, max_len=20):
    model.eval()
    features = model.encoder(image_tensor.unsqueeze(0).to(device))
    hidden, cell = model.decoder.init_hidden(features)
    token_id = vocab.start_idx
    words, attention_maps = [], []
    for _ in range(max_len):
        word_t = torch.tensor([token_id], device=device)
        context, alpha = model.decoder.attention(features, hidden)
        embedding = model.decoder.embed(word_t)
        hidden, cell = model.decoder.lstm_cell(torch.cat([embedding, context], 1), (hidden, cell))
        logits = model.decoder.fc(hidden)
        token_id = int(logits.argmax(-1).item())
        if token_id == vocab.end_idx: break
        if token_id in (vocab.pad_idx, vocab.start_idx): continue
        words.append(vocab.itos[token_id])
        attention_maps.append(alpha[0].detach().cpu().reshape(7, 7))
    return words, attention_maps

def export_token_attention(model, image_id, vocab, image_dir, id_to_file, transform, device, max_words=5):
    path = Path(image_dir) / id_to_file[image_id]
    img = Image.open(path).convert("RGB")
    inp = transform(img)
    words, weights = greedy_caption_with_attention(model, inp, vocab, device)
    if not weights:
        print("No attention maps available for this image.")
        return
    n = min(len(weights), max_words)
    fig, axes = plt.subplots(1, n + 1, figsize=(3.2*(n+1), 3.7))
    axes[0].imshow(img); axes[0].set_title("Original image")
    for j in range(n):
        a = F.interpolate(weights[j][None, None].float(), size=(img.height, img.width), mode="bilinear", align_corners=False).squeeze().numpy()
        axes[j+1].imshow(img)
        axes[j+1].imshow(a, alpha=0.55, cmap="inferno", vmin=0, vmax=max(1e-6, a.max()))
        axes[j+1].set_title(f"Token: {words[j]}")
    for ax in axes: ax.axis("off")
    fig.suptitle(f"RFP-Cap attention (greedy) | image_id={image_id}")
    fig.tight_layout()
    save_report_figure(fig, f"Figure_08_Token_Attention_{image_id}")
    return words

# Example after rerunning the full pipeline:
# selected_image_id = test_ids[0]
# generated_words = export_token_attention(model2, selected_image_id, vocab, IMG_DIR, id_to_file, eval_transform, device)
